In [16]:
import pandas as pd

In [3]:
import sys
!{sys.executable} -m pip install pandas


[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

In [128]:
df = pd.read_csv("OECD_RD_GDP_2000_2025.csv")
df2 = pd.read_csv("OECD_Public_Debt_GDP_2000_2025.csv")
df3 = pd.read_csv("OECD_Productivity_Variation_2000_2025.csv")
df4 = pd.read_csv("OECD_Tertiary_Education_2000_2025.csv")
df5 = pd.read_csv("OECD_Unemployment_Rate_2000_2025.csv")
df6 = pd.read_csv("OECD_Inflation_CPI_2000_2025.csv")
dft = pd.read_csv("OECD_Trust_Institutions_2000_2025.csv")
dfe = pd.read_csv("Eurostat_Energy_Import_Dependency_2000_2025.csv")
dfg = pd.read_csv("OECD_GDP_Growth_2000_2025.csv")
df_ge = pd.read_csv("Global_Energy_Dependency_0_100.csv")
df_eur = pd.read_csv("Eurostat_Public_Debt_GDP_2000_2025.csv")
df_gd = pd.read_csv("Global_Public_Debt_GDP_2000_2025.csv")


In [101]:
dfe

,Country,Year,Value
0,AL,2000,45.781
1,AL,2001,52.210
2,AL,2002,53.360
3,AL,2003,50.757
4,AL,2004,47.764
...,...,...,...
942,XK,2018,29.274
943,XK,2019,30.534
944,XK,2020,29.537
945,XK,2021,32.614


In [73]:
pip install pycountry

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [140]:
import pycountry

def convert_iso2_to_iso3(code):
    try:
        return pycountry.countries.get(alpha_2=code).alpha_3
    except:
        return None

In [141]:
dfe["Country"] = dfe["Country"].apply(convert_iso2_to_iso3)

In [131]:
dfe.head(30)

,Country,Year,Value
0,ALB,2000,45.781
1,ALB,2001,52.210
2,ALB,2002,53.360
3,ALB,2003,50.757
4,ALB,2004,47.764
5,ALB,2005,49.710
6,ALB,2006,40.387
7,ALB,2007,49.179
8,ALB,2008,49.780
9,ALB,2009,46.388


In [20]:
#Check inconsistent text
df["Country"] = df["Country"].str.lower().str.strip()

In [77]:
dfe

,Country,Year,Value
0,ALB,2000,45.781
1,ALB,2001,52.210
2,ALB,2002,53.360
3,ALB,2003,50.757
4,ALB,2004,47.764
...,...,...,...
942,NaN,2018,29.274
943,NaN,2019,30.534
944,NaN,2020,29.537
945,NaN,2021,32.614


In [146]:
import pandas as pd
import pycountry

files = [
    "OECD_RD_GDP_2000_2025.csv",
    "OECD_Productivity_Variation_2000_2025.csv",
    "OECD_Tertiary_Education_2000_2025.csv",
    "OECD_Unemployment_Rate_2000_2025.csv",
    "OECD_Inflation_CPI_2000_2025.csv",
    "OECD_Trust_Institutions_2000_2025.csv",
    "OECD_GDP_Growth_2000_2025.csv",
    "Global_Energy_Dependency_0_100.csv",
    "Eurostat_Public_Debt_GDP_2000_2025.csv",
    "Global_Public_Debt_GDP_2000_2025.csv"
]


# =========================
# COUNTRY CODE FIX
# =========================
def iso2_to_iso3(code):
    try:
        return pycountry.countries.get(alpha_2=code).alpha_3
    except:
        return None


# =========================
# CLEANING FUNCTION
# =========================
def clean_dataset(df, fix_iso2=False):

    df.columns = df.columns.str.lower().str.strip()

    df = df[["country", "year", "value"]]

    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    df = df.drop_duplicates()
    df = df.sort_values(["country", "year"])

    # 🔥 FIX ISO2 → ISO3 ONLY IF NEEDED
    if fix_iso2:
        df["country"] = df["country"].apply(iso2_to_iso3)

    df["value"] = df.groupby("country")["value"].transform(
        lambda x: x.interpolate().ffill().bfill()
    )

    return df


# =========================
# LOAD + CLEAN
# =========================
datasets = {}

for file in files:
    df = pd.read_csv(file)

    name = file.replace(".csv", "")

    # 🔥 detect Eurostat dataset
    if "Eurostat" in file:
        datasets[name] = clean_dataset(df, fix_iso2=True)
    else:
        datasets[name] = clean_dataset(df, fix_iso2=False)


# =========================
# PIVOT (SAFE VERSION)
# =========================
pivot_datasets = {}

for name, df in datasets.items():
    pivot_datasets[name] = df.pivot_table(
        index="year",
        columns="country",
        values="value",
        aggfunc="mean"
    ).sort_index()

In [147]:
df_ge

,country,Year,Value
0,AGO,2000,0.000000
1,AGO,2001,0.000000
2,AGO,2002,0.000000
3,AGO,2003,0.000000
4,AGO,2004,0.000000
...,...,...,...
3146,ZWE,2018,23.868661
3147,ZWE,2019,21.200162
3148,ZWE,2020,17.352023
3149,ZWE,2021,9.322581


In [148]:
#merging the datasets based on the year
# STEP 1: Convert each pivot dataset to LONG format
dfs_long = []

for name, df in pivot_datasets.items():
    
    temp = df.reset_index().melt(
        id_vars="year",
        var_name="country",
        value_name="value"
    )
    
    temp["indicator"] = name  # keep track of dataset source
    
    dfs_long.append(temp)


# STEP 2: Concatenate everything into one dataset
df_all = pd.concat(dfs_long, ignore_index=True)


# STEP 3: Pivot to final clean panel format
final_df = df_all.pivot_table(
    index=["year", "country"],
    columns="indicator",
    values="value"
).reset_index()




#Fill intelligently, NaN does NOT mean bad data, it means “data not available for that country-year-indicator”
final_clean = final_df.copy()

numeric_cols = final_clean.select_dtypes(include="number").columns

# 1. interpolate (time structure)
final_clean[numeric_cols] = final_clean.groupby("country")[numeric_cols].transform(
    lambda x: x.interpolate(limit_direction="both")
)

# 2. country mean
final_clean[numeric_cols] = final_clean.groupby("country")[numeric_cols].transform(
    lambda x: x.fillna(x.mean())
)




final_clean

indicator,year,country,Eurostat_Public_Debt_GDP_2000_2025,Global_Energy_Dependency_0_100,Global_Public_Debt_GDP_2000_2025,OECD_GDP_Growth_2000_2025,OECD_Inflation_CPI_2000_2025,OECD_Productivity_Variation_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Trust_Institutions_2000_2025,OECD_Unemployment_Rate_2000_2025
0,2000,AGO,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000,ALB,NaN,46.518154,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000,ARE,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000,ARG,NaN,0.000000,NaN,-0.788999,34.277230,NaN,0.392499,16.711861,NaN,7.561
4,2000,ARM,NaN,71.191142,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
3684,2025,SWE,35.1,27.752405,46.363,1.543167,0.679869,332.353013,3.559558,51.826313,42.4,8.398
3685,2025,TUR,NaN,71.992860,28.557,3.604625,34.881160,91.185172,1.461902,26.944220,NaN,8.712
3686,2025,USA,NaN,0.000000,123.186,2.106338,2.949525,90.335094,3.444858,50.705994,NaN,4.022
3687,2025,USMCA,NaN,NaN,NaN,1.931042,NaN,NaN,NaN,NaN,NaN,NaN


In [152]:

final_clean

indicator,year,country,Global_Energy_Dependency_0_100,Global_Public_Debt_GDP_2000_2025,OECD_GDP_Growth_2000_2025,OECD_Inflation_CPI_2000_2025,OECD_Productivity_Variation_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Trust_Institutions_2000_2025,OECD_Unemployment_Rate_2000_2025
0,2000,AGO,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000,ALB,46.518154,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000,ARE,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000,ARG,0.000000,NaN,-0.788999,34.277230,NaN,0.392499,16.711861,NaN,7.561
4,2000,ARM,71.191142,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
3684,2025,SWE,27.752405,46.363,1.543167,0.679869,332.353013,3.559558,51.826313,42.4,8.398
3685,2025,TUR,71.992860,28.557,3.604625,34.881160,91.185172,1.461902,26.944220,NaN,8.712
3686,2025,USA,0.000000,123.186,2.106338,2.949525,90.335094,3.444858,50.705994,NaN,4.022
3687,2025,USMCA,NaN,NaN,1.931042,NaN,NaN,NaN,NaN,NaN,NaN


In [164]:
missing_by_country = final_clean.groupby("country").apply(
    lambda x: x.isnull().sum()
).drop(columns=["year"]).reset_index()
missing_by_country

indicator,country,Global_Energy_Dependency_0_100,Global_Public_Debt_GDP_2000_2025,OECD_GDP_Growth_2000_2025,OECD_Inflation_CPI_2000_2025,OECD_Productivity_Variation_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Trust_Institutions_2000_2025,OECD_Unemployment_Rate_2000_2025
0,AGO,0,23,23,23,23,23,23,23,23
1,ALB,0,24,24,24,24,24,24,24,24
2,ARE,0,23,23,23,23,23,23,23,23
3,ARG,0,26,0,0,26,0,0,26,0
4,ARM,0,23,23,23,23,23,23,23,23
...,...,...,...,...,...,...,...,...,...,...
156,VNM,0,23,23,23,23,23,23,23,23
157,YEM,0,23,23,23,23,23,23,23,23
158,ZAF,0,0,0,0,26,0,0,26,0
159,ZMB,0,23,23,23,23,23,23,23,23


In [165]:
missing_by_country.to_csv("missing_by_country_4.csv", index=False)

In [166]:
!pip install pycountry


[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [167]:
#Create conversion function
import pycountry

def country_to_iso3(country_name):
    try:
        return pycountry.countries.lookup(country_name).alpha_3
    except:
        return None



In [168]:
df7 = pd.read_csv("TrustGov_2000_2020.csv")
df7.head()

,country,year,Approval_Not_Smoothed
0,Argentina,1983,67.252609
1,Argentina,1984,61.002979
2,Argentina,1985,58.382301
3,Argentina,1986,54.313831
4,Argentina,1987,48.490929


In [170]:
df7["country"] = df7["country"].apply(country_to_iso3)

In [171]:
df7 = df7[df7["year"] >= 2000]
df7

,country,year,Approval_Not_Smoothed
17,ARG,2000,46.135551
18,ARG,2001,31.172791
19,ARG,2002,30.174061
20,ARG,2003,59.634048
21,ARG,2004,59.264530
...,...,...,...
2215,VEN,2015,33.350620
2216,VEN,2016,29.990950
2217,VEN,2017,31.979071
2218,VEN,2018,30.125401


In [172]:
df7[df7["country"].isnull()]["country"].unique()

<StringArray>
[nan]
Length: 1, dtype: str

In [173]:
#some rows have no country name by the way
df7[df7["country"].isnull()]

,country,year,Approval_Not_Smoothed
1257,NaN,2007,52.485630
1258,NaN,2008,57.206451
1259,NaN,2009,48.144211
1260,NaN,2010,39.845322
1261,NaN,2011,34.870399
...,...,...,...
1941,NaN,2014,-0.521792
1942,NaN,2015,-1.468777
1943,NaN,2016,0.895465
1944,NaN,2017,0.838192


In [174]:
#Clening workflow
df7 = df7.dropna(subset=["country"])
df7["country_code"] = df7["country"].apply(country_to_iso3)

In [175]:
df7 = df7.rename(columns={"Approval_Not_Smoothed": "gov_trust"})

In [176]:
#merging
merged = pd.merge(
    final_clean,
    df7,
    on=["country", "year"],
    how="outer"
)

In [177]:
merged

,year,country,Global_Energy_Dependency_0_100,Global_Public_Debt_GDP_2000_2025,OECD_GDP_Growth_2000_2025,OECD_Inflation_CPI_2000_2025,OECD_Productivity_Variation_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Trust_Institutions_2000_2025,OECD_Unemployment_Rate_2000_2025,gov_trust,country_code
0,2000,AGO,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2001,AGO,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2002,AGO,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2003,AGO,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2004,AGO,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3970,2018,ZWE,23.868661,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3971,2019,ZWE,21.200162,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3972,2020,ZWE,17.352023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3973,2021,ZWE,9.322581,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [178]:
merged = merged.drop("country_code", axis=1)

In [179]:
merged

,year,country,Global_Energy_Dependency_0_100,Global_Public_Debt_GDP_2000_2025,OECD_GDP_Growth_2000_2025,OECD_Inflation_CPI_2000_2025,OECD_Productivity_Variation_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Trust_Institutions_2000_2025,OECD_Unemployment_Rate_2000_2025,gov_trust
0,2000,AGO,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2001,AGO,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2002,AGO,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2003,AGO,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2004,AGO,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
3970,2018,ZWE,23.868661,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3971,2019,ZWE,21.200162,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3972,2020,ZWE,17.352023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3973,2021,ZWE,9.322581,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
!pip install openpyxl


[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [180]:
dfx = pd.read_excel("Worldwide Governance Indicators.xlsx")

In [181]:
dfx.to_csv("your_file.csv", index=False)

In [182]:
dfx = dfx[dfx["Year"] >= 2000]
dfx

,ID variable (economy code/ gov. dimension/ year),Economy (name),Economy (code),Region,Income classification,Year,Governance dimension,Number of sources,Governance estimate (approx. -2.5 to +2.5),Standard error (estimate),...,OBI mean,PIA mean,PRS mean,RSF mean,VAB mean,VDM mean,WBS mean,WCY mean,WJP mean,WMO mean
403,ADOva2000,Andorra,ADO,NaN,NaN,2000,va,3,1.541361,0.301021,...,..,..,..,..,..,..,..,..,..,1
404,AFGva2000,Afghanistan,AFG,South Asia,Low income,2000,va,4,-2.297846,0.245080,...,..,..,..,..,..,0.245567,..,..,..,0.0625
405,AGOva2000,Angola,AGO,Sub-Saharan Africa,Lower middle income,2000,va,7,-1.678503,0.185910,...,..,..,0.333333,..,..,0.320267,..,..,..,0.125
406,ALBva2000,Albania,ALB,Europe & Central Asia,Upper middle income,2000,va,6,-0.524937,0.204243,...,..,..,0.75,..,..,0.460222,..,..,..,0.375
407,AREva2000,United Arab Emirates,ARE,Middle East & North Africa,High income,2000,va,6,-0.843748,0.193985,...,..,..,0.583333,..,..,0.335233,..,..,..,0.6875
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,XKXva2024,Kosovo,XKX,Europe & Central Asia,Upper middle income,2024,va,9,0.159588,0.156446,...,..,0.5,..,0.5273,..,0.607289,..,..,0.578919,..
5440,YEMva2024,"Yemen, Rep.",YEM,Middle East & North Africa,Low income,2024,va,9,-1.697182,0.178875,...,0,0.1,0.416667,0.3145,..,0.3571,..,..,..,..
5441,ZAFva2024,South Africa,ZAF,Sub-Saharan Africa,Upper middle income,2024,va,13,0.607280,0.147152,...,0.82713,..,0.833333,0.7571,..,0.650411,..,0.314754,0.619057,..
5442,ZMBva2024,Zambia,ZMB,Sub-Saharan Africa,Lower middle income,2024,va,14,-0.407569,0.142237,...,0.339358,0.3,0.791667,0.5733,..,0.5176,..,..,0.421383,..


In [183]:
dfx["Economy (name)"] = dfx["Economy (name)"].apply(country_to_iso3)

In [184]:
dfx

,ID variable (economy code/ gov. dimension/ year),Economy (name),Economy (code),Region,Income classification,Year,Governance dimension,Number of sources,Governance estimate (approx. -2.5 to +2.5),Standard error (estimate),...,OBI mean,PIA mean,PRS mean,RSF mean,VAB mean,VDM mean,WBS mean,WCY mean,WJP mean,WMO mean
403,ADOva2000,AND,ADO,NaN,NaN,2000,va,3,1.541361,0.301021,...,..,..,..,..,..,..,..,..,..,1
404,AFGva2000,AFG,AFG,South Asia,Low income,2000,va,4,-2.297846,0.245080,...,..,..,..,..,..,0.245567,..,..,..,0.0625
405,AGOva2000,AGO,AGO,Sub-Saharan Africa,Lower middle income,2000,va,7,-1.678503,0.185910,...,..,..,0.333333,..,..,0.320267,..,..,..,0.125
406,ALBva2000,ALB,ALB,Europe & Central Asia,Upper middle income,2000,va,6,-0.524937,0.204243,...,..,..,0.75,..,..,0.460222,..,..,..,0.375
407,AREva2000,ARE,ARE,Middle East & North Africa,High income,2000,va,6,-0.843748,0.193985,...,..,..,0.583333,..,..,0.335233,..,..,..,0.6875
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,XKXva2024,NaN,XKX,Europe & Central Asia,Upper middle income,2024,va,9,0.159588,0.156446,...,..,0.5,..,0.5273,..,0.607289,..,..,0.578919,..
5440,YEMva2024,NaN,YEM,Middle East & North Africa,Low income,2024,va,9,-1.697182,0.178875,...,0,0.1,0.416667,0.3145,..,0.3571,..,..,..,..
5441,ZAFva2024,ZAF,ZAF,Sub-Saharan Africa,Upper middle income,2024,va,13,0.607280,0.147152,...,0.82713,..,0.833333,0.7571,..,0.650411,..,0.314754,0.619057,..
5442,ZMBva2024,ZMB,ZMB,Sub-Saharan Africa,Lower middle income,2024,va,14,-0.407569,0.142237,...,0.339358,0.3,0.791667,0.5733,..,0.5176,..,..,0.421383,..


In [185]:
dfx.columns

Index(['ID variable (economy code/ gov. dimension/ year)', 'Economy (name)',
       'Economy (code)', 'Region', 'Income classification', 'Year',
       'Governance dimension', 'Number of sources',
       'Governance estimate (approx. -2.5 to +2.5)',
       'Standard error (estimate)',
       'Lower threshold (90% conf. int. estimate)',
       'Upper threshold (90% conf. int. estimate)', 'Governance score (0-100)',
       'Standard error (gov. score)', 'Lower threshold (90% conf. int. score)',
       'Upper threshold (90% conf. int. score)', 'ADB mean', 'AFR mean',
       'ARB mean', 'ASB mean', 'ASD mean', 'BPS mean', 'BTI mean', 'CCR mean',
       'EBR mean', 'EIU mean', 'EQI mean', 'EUB mean', 'FRH mean', 'GCB mean',
       'GCS mean', 'GII mean', 'GWP mean', 'HER mean', 'HRM mean', 'HUM mean',
       'IFD mean', 'IJT mean', 'IRP mean', 'LBO mean', 'MSI mean', 'OBI mean',
       'PIA mean', 'PRS mean', 'RSF mean', 'VAB mean', 'VDM mean', 'WBS mean',
       'WCY mean', 'WJP mean', 'WM

In [186]:
dfx = dfx.drop(['ID variable (economy code/ gov. dimension/ year)',
       'Economy (code)', 'Region', 'Income classification',
       'Governance dimension', 'Number of sources',
       'Governance estimate (approx. -2.5 to +2.5)',
       'Standard error (estimate)',
       'Lower threshold (90% conf. int. estimate)',
       'Upper threshold (90% conf. int. estimate)',
       'Standard error (gov. score)', 'Lower threshold (90% conf. int. score)',
       'Upper threshold (90% conf. int. score)', 'ADB mean', 'AFR mean',
       'ARB mean', 'ASB mean', 'ASD mean', 'BPS mean', 'BTI mean', 'CCR mean',
       'EBR mean', 'EIU mean', 'EQI mean', 'EUB mean', 'FRH mean', 'GCB mean',
       'GCS mean', 'GII mean', 'GWP mean', 'HER mean', 'HRM mean', 'HUM mean',
       'IFD mean', 'IJT mean', 'IRP mean', 'LBO mean', 'MSI mean', 'OBI mean',
       'PIA mean', 'PRS mean', 'RSF mean', 'VAB mean', 'VDM mean', 'WBS mean',
       'WCY mean', 'WJP mean', 'WMO mean'], axis=1)

In [187]:
dfx

,Economy (name),Year,Governance score (0-100)
403,AND,2000,83.826902
404,AFG,2000,16.631208
405,AGO,2000,27.471257
406,ALB,2000,47.661534
407,ARE,2000,42.081555
...,...,...,...
5439,NaN,2024,59.642428
5440,NaN,2024,27.144335
5441,ZAF,2024,67.478144
5442,ZMB,2024,49.715769


In [188]:
dfx.rename(columns={"Economy (name)": "country"}, inplace=True)
dfx.rename(columns={"Year": "year"}, inplace=True)

In [250]:
#merging
final_merged = pd.merge(
    final_clean,
    dfx,
    on=["country", "year"],
    how="inner"
)

In [251]:
final_merged

,year,country,Global_Energy_Dependency_0_100,Global_Public_Debt_GDP_2000_2025,OECD_GDP_Growth_2000_2025,OECD_Inflation_CPI_2000_2025,OECD_Productivity_Variation_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Trust_Institutions_2000_2025,OECD_Unemployment_Rate_2000_2025,Governance score (0-100)
0,2000,AGO,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.471257
1,2000,ALB,46.518154,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,47.661534
2,2000,ARE,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42.081555
3,2000,ARG,0.000000,NaN,-0.788999,34.277230,NaN,0.392499,16.711861,NaN,7.561,64.590212
4,2000,ARM,71.191142,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,46.781340
...,...,...,...,...,...,...,...,...,...,...,...,...
2856,2024,SVN,48.902182,69.318,1.731397,1.965627,48.513447,2.162628,34.637478,57.9,3.697,75.351985
2857,2024,SWE,27.752405,46.363,1.017797,2.835817,332.353013,3.559558,51.826313,42.4,8.398,87.089942
2858,2024,TUR,71.992860,28.557,3.327623,58.506450,91.185172,1.461902,26.944220,NaN,8.712,37.689790
2859,2024,USA,0.000000,123.186,2.793187,2.949525,90.335094,3.444858,50.705994,NaN,4.022,72.373172


In [242]:
missing_rows = final_clean[final_clean.isnull().any(axis=1)]
missing_rows.head(30)

indicator,year,country,Global_Energy_Dependency_0_100,Global_Public_Debt_GDP_2000_2025,OECD_GDP_Growth_2000_2025,OECD_Inflation_CPI_2000_2025,OECD_Productivity_Variation_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Trust_Institutions_2000_2025,OECD_Unemployment_Rate_2000_2025
0,2000,AGO,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000,ALB,46.518154,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000,ARE,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000,ARG,0.000000,NaN,-0.788999,34.277230,NaN,0.392499,16.711861,NaN,7.561000
4,2000,ARM,71.191142,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2000,AZE,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2000,BEN,28.097710,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,2000,BGD,17.650915,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,2000,BGR,46.864953,48.474,NaN,10.316260,11.549349,0.495167,27.498056,NaN,16.919000
12,2000,BHR,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [252]:
final_merged = final_merged.rename(columns={"OECD_GDP_Growth_2000_2025": "R&D expenditure (%GDP)"})
final_merged = final_merged.rename(columns={"OECD_Trust_Institutions_2000_2025": "trust_in_institutions"})
final_merged = final_merged.rename(columns={"Global_Public_Debt_GDP_2000_2025": "Public_Debt (%GDP)"})
final_merged = final_merged.rename(columns={"Global_Energy_Dependency_0_100": "Energy_Import_Dependency"})
final_merged = final_merged.rename(columns={"OECD_Tertiary_Education_2000_2025": "Tertiary_education_attainment"})

In [253]:
final_df1 = final_merged[
    [
        "year",
        "country",
        "R&D expenditure (%GDP)",
        "trust_in_institutions",
        "Public_Debt (%GDP)",
        "Energy_Import_Dependency",
        "Tertiary_education_attainment"
     
    ]
]

In [254]:
final_merged = final_merged.rename(columns={"OECD_Inflation_CPI_2000_2025": "Inflation_Rate"})
final_merged = final_merged.rename(columns={"OECD_Unemployment_Rate_2000_2025": "Unemployment_Variation"})
final_merged = final_merged.rename(columns={"Governance score (0-100)": "Gov_score"})
final_merged = final_merged.rename(columns={"OECD_Productivity_Variation_2000_2025": "Productivity_Variation"})
final_merged = final_merged.rename(columns={"R&D expenditure (%GDP)": "GDP_Growth"})
final_df2 = final_merged[
    [
        "year",
        "country",
        "Inflation_Rate",
        "Unemployment_Variation",
        "Gov_score",
        "Productivity_Variation",
        "GDP_Growth"
        
    ]
]

In [255]:
final_df1

,year,country,R&D expenditure (%GDP),trust_in_institutions,Public_Debt (%GDP),Energy_Import_Dependency,Tertiary_education_attainment
0,2000,AGO,NaN,NaN,NaN,0.000000,NaN
1,2000,ALB,NaN,NaN,NaN,46.518154,NaN
2,2000,ARE,NaN,NaN,NaN,0.000000,NaN
3,2000,ARG,-0.788999,NaN,NaN,0.000000,16.711861
4,2000,ARM,NaN,NaN,NaN,71.191142,NaN
...,...,...,...,...,...,...,...
2856,2024,SVN,1.731397,57.9,69.318,48.902182,34.637478
2857,2024,SWE,1.017797,42.4,46.363,27.752405,51.826313
2858,2024,TUR,3.327623,NaN,28.557,71.992860,26.944220
2859,2024,USA,2.793187,NaN,123.186,0.000000,50.705994


In [256]:
final_df2

,year,country,Inflation_Rate,Unemployment_Variation,Gov_score,Productivity_Variation,GDP_Growth
0,2000,AGO,NaN,NaN,27.471257,NaN,NaN
1,2000,ALB,NaN,NaN,47.661534,NaN,NaN
2,2000,ARE,NaN,NaN,42.081555,NaN,NaN
3,2000,ARG,34.277230,7.561,64.590212,NaN,-0.788999
4,2000,ARM,NaN,NaN,46.781340,NaN,NaN
...,...,...,...,...,...,...,...
2856,2024,SVN,1.965627,3.697,75.351985,48.513447,1.731397
2857,2024,SWE,2.835817,8.398,87.089942,332.353013,1.017797
2858,2024,TUR,58.506450,8.712,37.689790,91.185172,3.327623
2859,2024,USA,2.949525,4.022,72.373172,90.335094,2.793187


In [269]:
final_df3 = final_merged[
    [
        "year",
        "country",
        "Public_Debt (%GDP)",
        "Energy_Import_Dependency",
        "Tertiary_education_attainment",
        "Inflation_Rate",
        "Unemployment_Variation",
        "Gov_score",
        "Productivity_Variation",
        "GDP_Growth"
        
    ]
]

In [246]:
numeric_cols = final_df3.select_dtypes(include="number").columns

missing_by_country = final_df3.groupby("country")[numeric_cols].apply(
    lambda x: x.isnull().sum().sum()
).reset_index(name="missing_count")

final_df3 = final_df3.merge(missing_by_country, on="country", how="left")
final_df3

,year,country,Public_Debt (%GDP),Energy_Import_Dependency,Tertiary_education_attainment,Inflation_Rate,Unemployment_Variation,Gov_score,Productivity_Variation,GDP_Growth,missing_count
0,2000,AGO,NaN,0.000000,NaN,NaN,NaN,27.471257,NaN,NaN,132
1,2000,ALB,NaN,46.518154,NaN,NaN,NaN,47.661534,NaN,NaN,138
2,2000,ARE,NaN,0.000000,NaN,NaN,NaN,42.081555,NaN,NaN,132
3,2000,ARG,NaN,0.000000,16.711861,34.277230,7.561,64.590212,NaN,-0.788999,48
4,2000,ARM,NaN,71.191142,NaN,NaN,NaN,46.781340,NaN,NaN,132
...,...,...,...,...,...,...,...,...,...,...,...
2856,2024,SVN,69.318,48.902182,34.637478,1.965627,3.697,75.351985,48.513447,1.731397,0
2857,2024,SWE,46.363,27.752405,51.826313,2.835817,8.398,87.089942,332.353013,1.017797,0
2858,2024,TUR,28.557,71.992860,26.944220,58.506450,8.712,37.689790,91.185172,3.327623,0
2859,2024,USA,123.186,0.000000,50.705994,2.949525,4.022,72.373172,90.335094,2.793187,0


In [270]:
final_df3

,year,country,Public_Debt (%GDP),Energy_Import_Dependency,Tertiary_education_attainment,Inflation_Rate,Unemployment_Variation,Gov_score,Productivity_Variation,GDP_Growth
0,2000,AGO,NaN,0.000000,NaN,NaN,NaN,27.471257,NaN,NaN
1,2000,ALB,NaN,46.518154,NaN,NaN,NaN,47.661534,NaN,NaN
2,2000,ARE,NaN,0.000000,NaN,NaN,NaN,42.081555,NaN,NaN
3,2000,ARG,NaN,0.000000,16.711861,34.277230,7.561,64.590212,NaN,-0.788999
4,2000,ARM,NaN,71.191142,NaN,NaN,NaN,46.781340,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
2856,2024,SVN,69.318,48.902182,34.637478,1.965627,3.697,75.351985,48.513447,1.731397
2857,2024,SWE,46.363,27.752405,51.826313,2.835817,8.398,87.089942,332.353013,1.017797
2858,2024,TUR,28.557,71.992860,26.944220,58.506450,8.712,37.689790,91.185172,3.327623
2859,2024,USA,123.186,0.000000,50.705994,2.949525,4.022,72.373172,90.335094,2.793187


In [259]:
final_df3

,year,country,Public_Debt (%GDP),Energy_Import_Dependency,Tertiary_education_attainment,Inflation_Rate,Unemployment_Variation,Gov_score,Productivity_Variation,GDP_Growth
0,2000,AGO,NaN,0.000000,NaN,NaN,NaN,27.471257,NaN,NaN
1,2000,ALB,NaN,46.518154,NaN,NaN,NaN,47.661534,NaN,NaN
2,2000,ARE,NaN,0.000000,NaN,NaN,NaN,42.081555,NaN,NaN
3,2000,ARG,NaN,0.000000,16.711861,34.277230,7.561,64.590212,NaN,-0.788999
4,2000,ARM,NaN,71.191142,NaN,NaN,NaN,46.781340,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
2856,2024,SVN,69.318,48.902182,34.637478,1.965627,3.697,75.351985,48.513447,1.731397
2857,2024,SWE,46.363,27.752405,51.826313,2.835817,8.398,87.089942,332.353013,1.017797
2858,2024,TUR,28.557,71.992860,26.944220,58.506450,8.712,37.689790,91.185172,3.327623
2859,2024,USA,123.186,0.000000,50.705994,2.949525,4.022,72.373172,90.335094,2.793187


In [271]:
final_df3 = final_df3[final_df3.isnull().sum(axis=1) <= 6]

In [272]:
final_df3

,year,country,Public_Debt (%GDP),Energy_Import_Dependency,Tertiary_education_attainment,Inflation_Rate,Unemployment_Variation,Gov_score,Productivity_Variation,GDP_Growth
0,2000,AGO,NaN,0.000000,NaN,NaN,NaN,27.471257,NaN,NaN
1,2000,ALB,NaN,46.518154,NaN,NaN,NaN,47.661534,NaN,NaN
2,2000,ARE,NaN,0.000000,NaN,NaN,NaN,42.081555,NaN,NaN
3,2000,ARG,NaN,0.000000,16.711861,34.277230,7.561,64.590212,NaN,-0.788999
4,2000,ARM,NaN,71.191142,NaN,NaN,NaN,46.781340,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
2856,2024,SVN,69.318,48.902182,34.637478,1.965627,3.697,75.351985,48.513447,1.731397
2857,2024,SWE,46.363,27.752405,51.826313,2.835817,8.398,87.089942,332.353013,1.017797
2858,2024,TUR,28.557,71.992860,26.944220,58.506450,8.712,37.689790,91.185172,3.327623
2859,2024,USA,123.186,0.000000,50.705994,2.949525,4.022,72.373172,90.335094,2.793187


In [268]:
final_df3.to_csv("the_fulldataset_fixed", index=False)

In [214]:
final_df1.to_csv("first_separated2", index=False)
final_df2.to_csv("second_separated2", index=False)

In [109]:
col = "Energy_Import_Dependency"
missing_countries = final_df3.loc[
    final_df3[col].isnull(),
    ["country", col]
].drop_duplicates()

In [114]:
missing_countries

,country,Energy_Import_Dependency
24,ARG,NaN
48,AUS,NaN
191,BOL,NaN
209,BRA,NaN
233,CAN,NaN
257,CHE,NaN
281,CHL,NaN
305,CHN,NaN
329,COL,NaN
353,CRI,NaN


In [115]:
missing_countries.to_csv("countries_with _missingvalues_ENERGYDATASET.csv", index=False)

In [116]:
col = "Public_Debt (%GDP)"
missing_countries2 = final_df3.loc[
    final_df3[col].isnull(),
    ["country", col]
].drop_duplicates()
missing_countries2

,country,Public_Debt (%GDP)
0,ALB,NaN
24,ARG,NaN
180,BIH,NaN
191,BOL,NaN
305,CHN,NaN
353,CRI,NaN
377,CYP,NaN
527,DOM,NaN
546,ECU,NaN
722,GEO,NaN


In [117]:
missing_countries2.to_csv("countries_with _missingvalues_PublicDebtGDP.csv", index=False)

In [93]:
final_df3.to_csv("completedataset2.csv", index=False)

In [314]:
#to find where saved
import os
os.getcwd()

'C:\\Users\\21626\\Desktop\\ds_lab'

In [51]:
import numpy as np

# function to calculate recovery time
def calculate_recovery_time(df, shock_year, gdp_col):

    recovery_times = []

    for country in df["country"].unique():

        country_df = df[df["country"] == country].sort_values("year")

        # get GDP value before shock
        try:
            baseline = country_df.loc[
                country_df["year"] == shock_year,
                gdp_col
            ].values[0]
        except:
            continue

        recovery_year = np.nan

        # search future years
        future_data = country_df[country_df["year"] > shock_year]

        for _, row in future_data.iterrows():

            if row[gdp_col] >= baseline:
                recovery_year = row["year"] - shock_year
                break

        # assign result to all rows of that country
        country_df = country_df.copy()
        country_df[f"recovery_{shock_year}"] = recovery_year

        recovery_times.append(country_df)

    return pd.concat(recovery_times)


# =========================
# APPLY FOR 2007
# =========================
final_df2 = calculate_recovery_time(
    final_df2,
    shock_year=2007,
    gdp_col="GDP_Growth"
)

# =========================
# APPLY FOR 2019
# =========================
final_df2 = calculate_recovery_time(
    final_df2,
    shock_year=2019,
    gdp_col="GDP_Growth"
)

In [56]:
final_df2.head(100)

,year,country,Inflation_Rate,Unemployment_Variation,Gov_score,Productivity_Variation,GDP_Growth
0,2000,ALB,NaN,NaN,47.661534,NaN,NaN
1,2002,ALB,NaN,NaN,53.080731,NaN,NaN
2,2003,ALB,NaN,NaN,54.631023,NaN,NaN
3,2004,ALB,NaN,NaN,56.665036,NaN,NaN
4,2005,ALB,NaN,NaN,58.610539,NaN,NaN
...,...,...,...,...,...,...,...
95,2018,AUT,1.998382,4.848,82.479855,67.608703,2.484221
96,2019,AUT,1.530895,4.487,82.317535,68.750874,1.754976
97,2020,AUT,1.381909,5.363,83.628517,68.190766,-6.318255
98,2021,AUT,2.766667,6.181,82.945515,72.742376,4.923092


In [56]:
final_df1.to_csv("finalfinalfinal_dataset_one.csv", index=False)

final_df2.to_csv("finalfinalfinal_dataset_two.csv", index=False)

In [298]:
df12 = pd.read_csv("final_dataset_all_12_columns.csv")

In [299]:
df12

,year,country,Eurostat_Public_Debt_GDP_2000_2025,Global_Energy_Dependency_0_100,Global_Public_Debt_GDP_2000_2025,OECD_GDP_Growth_2000_2025,OECD_Inflation_CPI_2000_2025,OECD_Productivity_Variation_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Unemployment_Rate_2000_2025,Worldwide_Governance_Indicators
0,2000,AGO,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.471257
1,2000,ALB,NaN,46.518154,NaN,NaN,NaN,NaN,NaN,NaN,NaN,47.661534
2,2000,ARE,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42.081555
3,2000,ARG,NaN,0.000000,NaN,-0.788999,34.277230,NaN,0.392499,16.711861,7.561,64.590212
4,2000,ARM,NaN,71.191142,NaN,NaN,NaN,NaN,NaN,NaN,NaN,46.781340
...,...,...,...,...,...,...,...,...,...,...,...,...
2856,2024,SVN,66.4,48.902182,69.318,1.731397,1.965627,48.513447,2.162628,34.637478,3.697,75.351985
2857,2024,SWE,34.2,27.752405,46.363,1.017797,2.835817,332.353013,3.559558,51.826313,8.398,87.089942
2858,2024,TUR,NaN,71.992860,28.557,3.327623,58.506450,91.185172,1.461902,26.944220,8.712,37.689790
2859,2024,USA,NaN,0.000000,123.186,2.793187,2.949525,90.335094,3.444858,50.705994,4.022,72.373172


In [316]:
# ==============================
# WORKING DATASET
# ==============================
df = df12.copy()

# ==============================
# STEP 1: Identify numeric columns
# ==============================
numeric_cols = df.select_dtypes(include="number").columns

# ==============================
# STEP 2: Compute missing ratio per country
# ==============================
missing_ratio = df.groupby("country")[numeric_cols].apply(
    lambda x: x.isnull().mean().mean()
)

# ==============================
# STEP 3: Keep countries with acceptable missingness
# ==============================
good_countries = missing_ratio[missing_ratio < 0.30].index
df_clean = df[df["country"].isin(good_countries)].copy()

# ==============================
# STEP 4: COUNTRY-LEVEL MEAN IMPUTATION
# (fill NaNs using mean of same country)
# ==============================
df_clean[numeric_cols] = df_clean.groupby("country")[numeric_cols].transform(
    lambda x: x.fillna(x.mean())
)

# ==============================
# STEP 5: GLOBAL MEAN IMPUTATION (rounded to 2 decimals)
# ==============================
global_means = df_clean[numeric_cols].mean().round(2)

df_clean[numeric_cols] = df_clean[numeric_cols].fillna(global_means)

# ==============================
# STEP 6: CHECK RESULT
# ==============================
df_clean.head(50)

,year,country,Eurostat_Public_Debt_GDP_2000_2025,Global_Energy_Dependency_0_100,Global_Public_Debt_GDP_2000_2025,OECD_GDP_Growth_2000_2025,OECD_Inflation_CPI_2000_2025,OECD_Productivity_Variation_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Unemployment_Rate_2000_2025,Worldwide_Governance_Indicators
3,2000,ARG,56.44,0.000000,68.830,-0.788999,34.277230,739.790000,0.392499,16.711861,7.561000,64.590212
5,2000,AUS,56.44,0.000000,12.765,3.359670,4.457435,52.464076,1.469756,27.475746,6.288000,82.051731
6,2000,AUT,66.60,66.900318,70.344,3.189522,2.344868,47.091146,1.896676,24.783438,3.502000,81.175534
8,2000,BEL,109.70,87.219960,94.929,3.716706,2.544518,53.979823,1.936195,27.084993,7.009000,82.118608
11,2000,BGR,70.70,46.864953,48.474,2.610000,10.316260,11.549349,0.495167,27.498056,16.919000,62.642856
16,2000,BRA,56.44,23.641203,87.124,4.387949,7.044141,739.790000,1.640000,9.623828,9.358000,61.899754
18,2000,CAN,56.44,0.000000,71.325,5.138539,2.719440,45.872960,1.858470,40.105782,6.821000,85.196202
19,2000,CHE,56.44,56.694553,45.939,3.633217,1.558529,61.589523,2.238910,24.177282,2.657000,84.156770
20,2000,CHL,56.44,70.494045,11.376,4.971621,3.843273,3061.758841,0.311431,17.120726,10.481000,75.183648
21,2000,CHN,56.44,2.458075,68.830,8.600000,0.400000,739.790000,0.884096,9.681174,3.576000,28.962658


In [317]:
df_clean.to_csv("final_dataset_all_12_columns_fixed_missing_values.csv", index=False)

In [318]:
df_clean

,year,country,Eurostat_Public_Debt_GDP_2000_2025,Global_Energy_Dependency_0_100,Global_Public_Debt_GDP_2000_2025,OECD_GDP_Growth_2000_2025,OECD_Inflation_CPI_2000_2025,OECD_Productivity_Variation_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Unemployment_Rate_2000_2025,Worldwide_Governance_Indicators
3,2000,ARG,56.44,0.000000,68.830,-0.788999,34.277230,739.790000,0.392499,16.711861,7.561,64.590212
5,2000,AUS,56.44,0.000000,12.765,3.359670,4.457435,52.464076,1.469756,27.475746,6.288,82.051731
6,2000,AUT,66.60,66.900318,70.344,3.189522,2.344868,47.091146,1.896676,24.783438,3.502,81.175534
8,2000,BEL,109.70,87.219960,94.929,3.716706,2.544518,53.979823,1.936195,27.084993,7.009,82.118608
11,2000,BGR,70.70,46.864953,48.474,2.610000,10.316260,11.549349,0.495167,27.498056,16.919,62.642856
...,...,...,...,...,...,...,...,...,...,...,...,...
2856,2024,SVN,66.40,48.902182,69.318,1.731397,1.965627,48.513447,2.162628,34.637478,3.697,75.351985
2857,2024,SWE,34.20,27.752405,46.363,1.017797,2.835817,332.353013,3.559558,51.826313,8.398,87.089942
2858,2024,TUR,56.44,71.992860,28.557,3.327623,58.506450,91.185172,1.461902,26.944220,8.712,37.689790
2859,2024,USA,56.44,0.000000,123.186,2.793187,2.949525,90.335094,3.444858,50.705994,4.022,72.373172
